# Inspect the Cevrim objects

Read-only tour of what is on disk. **No cell writes to any `.h5ad`.**

Everything opens `backed='r'`, so `.obs`, shapes and `.obsm` load without pulling
expression matrices into memory. Only the last section (commented out) loads an
object fully.

**Kernel:** `Python (rsc_uv)` — see the setup notes at the bottom if it is missing.

### Contents

1. Inventory of every candidate object
2. The clobbered human All_Cells file
3. Current label state on the rebuild source
4. The two known label bugs
5. Preview of the rebuild (writes nothing)

## 0. Setup

In [1]:
import anndata as ad
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 200)

In [2]:
REPO       = Path("/n/lab_storage/eddy_lab/users/nhilgert/xmens_final")
NETSCRATCH = Path("/n/netscratch/eddy_lab/Everyone/nhilgert")

OBJ = {
    # --- the four current deliverables (on PURGEABLE scratch) ---
    "08_21 Human_All  (SUSPECT)": NETSCRATCH / "08_21/2026_08_21_Cevrim_Human_All_Cells.h5ad",
    "08_21 Human_FibDec":         NETSCRATCH / "08_21/2026_08_21_Cevrim_Human_Only_Fib_Dec_Cells.h5ad",
    "08_21 XMens_All":            NETSCRATCH / "08_21/2026_08_21_Cevrim_XMens_All_Cells.h5ad",
    "08_21 XMens_FibDec":         NETSCRATCH / "08_21/2026_08_21_Cevrim_XMens_Only_Fib_Dec_Cells.h5ad",

    # --- the file suspected of overwriting the human deliverable ---
    "MYSTERY allcells.h5ad":      NETSCRATCH / "allcells.h5ad",

    # --- pipeline outputs: the rebuild source ---
    "PIPE refined_human":         REPO / "data/interim/refined_human.h5ad",
    "PIPE refined_mouse":         REPO / "data/interim/refined_mouse.h5ad",
    "PIPE final_human":           REPO / "data/interim/final_human.h5ad",
    "PIPE final_mouse":           REPO / "data/interim/final_mouse.h5ad",
}


def peek(path):
    """Open an h5ad backed='r'. Never loads X into memory."""
    return ad.read_h5ad(path, backed="r")


for name, path in OBJ.items():
    status = "OK     " if path.exists() else "MISSING"
    print(f"{status}  {name}")

OK       08_21 Human_All  (SUSPECT)
OK       08_21 Human_FibDec
OK       08_21 XMens_All
OK       08_21 XMens_FibDec
OK       MYSTERY allcells.h5ad
OK       PIPE refined_human
OK       PIPE refined_mouse
OK       PIPE final_human
OK       PIPE final_mouse


## 1. Inventory

Expected cell counts: **human 66,830**, **mouse 57,033**.

Any `All_Cells` row that disagrees is wrong.

In [3]:
rows = []

for name, path in OBJ.items():
    if not path.exists():
        continue

    a = peek(path)
    obs = a.obs.columns

    rows.append({
        "object":       name,
        "n_obs":        a.n_obs,
        "n_vars":       a.n_vars,
        "GB":           round(path.stat().st_size / 1e9, 2),
        "obs_cols":     len(obs),
        "n_auc":        sum(c.startswith("auc_") for c in obs),
        "cell_type":    "cell_type" in obs,
        "cell_subtype": "cell_subtype" in obs,
        "dbscan":       "x_um_dbscan" in obs,
        "stro7":        "stro7_margin" in obs,
    })

    a.file.close()

inventory = pd.DataFrame(rows)
inventory

,object,n_obs,n_vars,GB,obs_cols,n_auc,cell_type,cell_subtype,dbscan,stro7
0,08_21 Human_All (SUSPECT),47533,36601,1.53,14,0,True,True,True,True
1,08_21 Human_FibDec,47533,36601,1.53,13,0,True,True,True,False
2,08_21 XMens_All,57033,33698,1.98,21,11,True,True,True,False
3,08_21 XMens_FibDec,43197,33698,1.54,24,11,True,True,True,False
4,MYSTERY allcells.h5ad,47533,36601,1.53,14,0,True,True,True,True
5,PIPE refined_human,66830,36601,2.09,10,0,True,False,False,False
6,PIPE refined_mouse,57033,33698,1.85,6,0,False,False,False,False
7,PIPE final_human,66830,36601,2.24,31,0,True,False,False,False
8,PIPE final_mouse,57033,33698,1.98,27,0,False,False,False,False


## 2. The clobbered file

`08_21 Human_All` should hold **66,830** cells across six cell types.

If it shows **47,533** cells with only `endometrial_fibroblast` and `decidual_cell`,
it is the Fib/Dec object under the wrong name.

`allcells.h5ad` is byte-identical in size and carries the same `stro7_margin`
column, which makes it the likely source of the overwrite.

In [4]:
suspects = [
    "08_21 Human_All  (SUSPECT)",
    "08_21 Human_FibDec",
    "MYSTERY allcells.h5ad",
]

for name in suspects:
    a = peek(OBJ[name])

    print(f"\n{name}")
    print(f"   n_obs        : {a.n_obs:,}")
    print(f"   size         : {OBJ[name].stat().st_size:,} bytes")
    print(f"   stro7_margin : {'stro7_margin' in a.obs.columns}")

    if "cell_type" in a.obs.columns:
        counts = a.obs["cell_type"].astype(str).value_counts()
        print("   cell_type:")
        for label, n in counts.items():
            print(f"      {label:28s} {n:>8,}")

    a.file.close()


08_21 Human_All  (SUSPECT)
   n_obs        : 47,533
   size         : 1,531,084,477 bytes
   stro7_margin : True
   cell_type:
      endometrial_fibroblast         25,885
      decidual_cell                  21,648

08_21 Human_FibDec
   n_obs        : 47,533
   size         : 1,530,703,821 bytes
   stro7_margin : False
   cell_type:
      endometrial_fibroblast         25,885
      decidual_cell                  21,648

MYSTERY allcells.h5ad
   n_obs        : 47,533
   size         : 1,531,084,477 bytes
   stro7_margin : True
   cell_type:
      endometrial_fibroblast         25,885
      decidual_cell                  21,648


## 3. Current label state on the rebuild source

`refined_*` is where the rebuild starts.

Note the two objects do **not** share an obs schema — human carries
`lineage` / `cell_type` / `subtype` / `annotation_level`, mouse carries none of
them. Both do have `coarse_refined` and `fine_refined`, which is what the
vocabulary maps from.

In [5]:
for species in ("human", "mouse"):
    a = peek(OBJ[f"PIPE refined_{species}"])

    print("=" * 70)
    print(f"refined_{species}   n_obs = {a.n_obs:,}")
    print("=" * 70)
    print("obs columns:")
    for col in a.obs.columns:
        print(f"   {col}")

    a.file.close()
    print()

refined_human   n_obs = 66,830
obs columns:
   sample
   cell_barcode
   timepoint
   coarse_refined
   fine_refined
   origin_lineage
   lineage
   cell_type
   subtype
   annotation_level

refined_mouse   n_obs = 57,033
obs columns:
   sample
   cell_barcode
   timepoint
   coarse_refined
   fine_refined
   origin_lineage



In [6]:
species = "mouse"      # flip to "human"

a = peek(OBJ[f"PIPE refined_{species}"])

for col in ("coarse_refined", "fine_refined"):
    counts = a.obs[col].astype(str).value_counts()
    print(f"\n--- {col}  ({len(counts)} values) ---")
    for label, n in counts.items():
        print(f"   {label:34s} {n:>8,}")

a.file.close()


--- coarse_refined  (7 values) ---
   decidual cells                       22,919
   endometrial fibroblasts              20,278
   epithelial cells                      6,265
   vascular cells                        3,487
   immune cells                          2,091
   smooth muscle cells                   1,443
   perimetrium cells                       550

--- fine_refined  (20 values) ---
   dec2                                 14,434
                                        11,745
   eFib2i                                8,775
   eFib2                                 4,764
   eFib3                                 3,886
   dec3                                  3,856
   dec1                                  3,722
   eFib1                                 2,073
   decAp                                   907
   Macrophages                             797
   eFibP                                   780
   Gzmb lo. Itgax hi. uNK cells            436
   Monocytes                        

## 4. The two label bugs

### 4a. Mouse endothelial cells were coarse-filled

The cleanup notebook coarse-filled **11,745** mouse cells.

The vocabulary sanctions **8,258**:

| lineage | count |
|---|---|
| epithelial | 6,265 |
| smooth muscle | 1,443 |
| perimetrium | 550 |
| **total** | **8,258** |

The excess is exactly **3,487** — the endothelial cells. They should carry
`cell_subtype = endothelial_blood`, but with no fine mapping for
`vascular cells` they fell through to the coarse fill.

In [7]:
a = peek(OBJ["08_21 XMens_All"])

labels = pd.DataFrame({
    "cell_type":    a.obs["cell_type"].astype(str),
    "cell_subtype": a.obs["cell_subtype"].astype(str),
})

a.file.close()

is_filled = labels.cell_subtype == labels.cell_type

summary = pd.crosstab(labels.cell_type, is_filled)
summary.columns = ["has_fine_subtype", "coarse_filled"]

print(summary.to_string())
print(f"\ntotal coarse-filled : {int(is_filled.sum()):,}")
print(f"vocabulary expects  : 8,258")
print(f"excess              : {int(is_filled.sum()) - 8258:,}")

                        has_fine_subtype  coarse_filled
cell_type                                              
decidual_cell                      22919              0
endometrial_fibroblast             20278              0
endothelial_cell                       0           3487
epithelial_cell                        0           6265
immune_cell                         2091              0
mesothelial_cell                       0            550
smooth_muscle_cell                     0           1443

total coarse-filled : 11,745
vocabulary expects  : 8,258
excess              : 3,487


In [8]:
endo = labels[labels.cell_type == "endothelial_cell"]

print("cell_subtype on mouse endothelial cells:")
print(endo.cell_subtype.value_counts().to_string())
print("\nshould be: endothelial_blood for all of them")

cell_subtype on mouse endothelial cells:
cell_subtype
endothelial_cell    3487

should be: endothelial_blood for all of them


### 4b. Human endothelial subtypes are orphaned

`endothelial_blood` and `endothelial_lymph` appear in `cell_subtype` while
`endothelial_cell` is **absent** from `cell_type` — children with no parent.

That is the missing perivascular split: 1,334 cells should become
`endothelial_cell` and 471 should stay `perivascular_cell`.

*(The file currently under that name is the Fib/Dec object, so this cell will
only show the two stromal types. The check matters for the rebuilt object.)*

In [9]:
a = peek(OBJ["08_21 Human_All  (SUSPECT)"])

cell_types    = set(a.obs["cell_type"].astype(str).unique())
cell_subtypes = set(a.obs["cell_subtype"].astype(str).unique())

a.file.close()

print("cell_type:")
for t in sorted(cell_types):
    print(f"   {t}")

print("\ncell_subtype:")
for t in sorted(cell_subtypes):
    print(f"   {t}")

endothelial = sorted(s for s in cell_subtypes if s.startswith("endothelial"))
print(f"\nendothelial subtypes present : {endothelial}")
print(f"endothelial_cell in cell_type: {'endothelial_cell' in cell_types}")

cell_type:
   decidual_cell
   endometrial_fibroblast

cell_subtype:
   decidual_cell
   endometrial_fibroblast

endothelial subtypes present : []
endothelial_cell in cell_type: False


## 5. Preview the rebuild

Applies the **v2** vocabulary to `refined_*` in memory and reports what the
deliverable would contain.

**Writes nothing.** Install the v2 YAML first, or point `VOCAB_PATH` at your
review copy.

In [11]:
import yaml

VOCAB_PATH = REPO / "resources/annotation_vocabulary.yaml"
VOCAB_PATH = REPO / "resources/annotation_vocabulary_v2.yaml"

vocab = yaml.safe_load(VOCAB_PATH.read_text())
print("schema:", vocab["schema_version"])

assert vocab["schema_version"].endswith("v2"), (
    "This is still the v1 vocabulary — install the corrected file first."
)

schema: xmens.annotation.v2


In [12]:
SPECIES = "mouse"      # flip to "human" to preview the perivascular split

a = peek(OBJ[f"PIPE refined_{SPECIES}"])
coarse = a.obs["coarse_refined"].astype(str)
fine   = a.obs["fine_refined"].astype(str)
a.file.close()

mapping = vocab["import_mapping"][SPECIES]

# --- 1. coarse rename ---
cell_type = coarse.map(lambda v: mapping["cell_type"].get(v, v))

# --- 2. content tasks (NOT renames) ---
cell_subtype = pd.Series([""] * len(cell_type), index=cell_type.index, dtype=object)

if SPECIES == "mouse":
    # every vascular cell is blood endothelium
    cell_subtype[(coarse == "vascular cells").values] = "endothelial_blood"
else:
    peri     = (coarse == "perivascular cells").values
    is_blood = peri & (fine == "blood endothelium").values
    is_lymph = peri & (fine == "lymphatic endothelium").values
    is_mural = peri & ~(is_blood | is_lymph)

    cell_type[is_blood | is_lymph] = "endothelial_cell"
    cell_type[is_mural]            = "perivascular_cell"

    cell_subtype[is_blood] = "endothelial_blood"
    cell_subtype[is_lymph] = "endothelial_lymph"
    cell_subtype[is_mural] = "perivascular_cell"

# --- 3. fine rename ---
unset = (cell_subtype == "").values
cell_subtype[unset] = fine.map(lambda v: mapping["cell_subtype"].get(v, "")).values[unset]

# --- 4. coarse fill ---
empty = cell_subtype.astype(str).isin(["", "nan", "None"]).values
n_filled = int(empty.sum())
cell_subtype[empty] = cell_type.values[empty]

expected_fill = vocab["content_tasks"]["coarse_subtype_fill"]["expected_total_fills"][SPECIES]

print(f"coarse-filled : {n_filled:,}")
print(f"expected      : {expected_fill:,}")
print(f"match         : {n_filled == expected_fill}")

coarse-filled : 8,258
expected      : 8,258
match         : True


In [13]:
print("cell_type:")
print(cell_type.value_counts().to_string())

print("\ncell_subtype:")
print(cell_subtype.astype(str).value_counts().to_string())

cell_type:
coarse_refined
decidual_cell             22919
endometrial_fibroblast    20278
epithelial_cell            6265
endothelial_cell           3487
immune_cell                2091
smooth_muscle_cell         1443
perimetrial_cell            550

cell_subtype:
decidual_proliferative          14434
efib_middle_induced              8775
epithelial_cell                  6265
efib_middle                      4764
efib_subluminal                  3886
decidual_mature                  3856
decidual_boundary                3722
endothelial_blood                3487
efib_outer                       2073
smooth_muscle_cell               1443
decidual_apoptotic                907
macrophage                        797
efib_proliferative                780
perimetrial_cell                  550
unk_cell_gzmb_low_itgax_high      436
monocyte                          204
unk_cell_proliferative            154
macrophage_proliferative          150
t_cell                             97
dendritic_cel

### Gate check — do the counts match the vocabulary?

In [14]:
expected = vocab["expected_coarse_counts"][SPECIES]
actual   = cell_type.value_counts().to_dict()

gate = pd.DataFrame([
    {
        "cell_type": label,
        "expected":  n,
        "got":       int(actual.get(label, 0)),
        "match":     int(actual.get(label, 0)) == n,
    }
    for label, n in expected.items()
])

print(gate.to_string(index=False))
print(f"\nall gates pass: {bool(gate['match'].all())}")

             cell_type  expected   got  match
         decidual_cell     22919 22919   True
endometrial_fibroblast     20278 20278   True
       epithelial_cell      6265  6265   True
           immune_cell      2091  2091   True
    smooth_muscle_cell      1443  1443   True
      endothelial_cell      3487  3487   True
      perimetrial_cell       550   550   True

all gates pass: True


### Hierarchy check — orphans and illegal pairs

In [15]:
allowed = vocab["allowed"]

pairs = pd.DataFrame({
    "cell_type":    cell_type.astype(str),
    "cell_subtype": cell_subtype.astype(str),
})

# a subtype must be allowed under its parent, or equal it (coarse fill)
illegal = sorted({
    (parent, child)
    for parent, child in zip(pairs.cell_type, pairs.cell_subtype)
    if child != parent and child not in set(allowed.get(parent) or [])
})

# a child present whose parent is absent from cell_type
present = set(pairs.cell_type)
orphans = [
    (child, parent)
    for parent, children in allowed.items()
    for child in (children or [])
    if child in set(pairs.cell_subtype) and parent not in present
]

n_empty = int(pairs.cell_subtype.isin(["", "nan", "None"]).sum())

print(f"illegal parent/child pairs : {illegal or 'NONE'}")
print(f"orphaned subtypes          : {orphans or 'NONE'}")
print(f"empty cell_subtype         : {n_empty}")

illegal parent/child pairs : NONE
orphaned subtypes          : NONE
empty cell_subtype         : 0


## 6. Optional — load an object fully

Only on an allocation with room (**64 GB+**). Everything above is backed and cheap.

`backed='r'` gives you `.obs`, `.var`, shapes and `.obsm`. You need a full load
only for expression values or plotting.

In [ ]:
# a = ad.read_h5ad(OBJ["PIPE refined_mouse"])   # ~1.9 GB on disk, more in RAM
# a

---

## Setup notes

### Launching Jupyter on Cannon

FASRC Open OnDemand: <https://rcood.rc.fas.harvard.edu> → Interactive Apps → Jupyter

Request **partition `eddy`**, **64 GB**, 4 cores, a couple of hours.
Do not run this on a login node.

### Registering the rsc_uv kernel

If `Python (rsc_uv)` is not in the kernel dropdown, run once from a terminal:

```bash
source /n/lab_storage/eddy_lab/users/nhilgert/.uv/envs/rsc_uv/bin/activate
python -m ipykernel install --user --name rsc_uv --display-name "Python (rsc_uv)"
```

`uva rsc_uv` does the same activation interactively. It is a `.bashrc` function,
so it does **not** exist in non-interactive job scripts — those need the absolute
path above.

### If the kernel dies

Almost always memory. Stay with `backed='r'`, or request a larger allocation.

Keep caches off HOME as well — it runs 90-95% full, and a full HOME makes Python
wedge at low CPU rather than fail cleanly:

```bash
export XDG_CACHE_HOME=/n/lab_storage/eddy_lab/users/nhilgert/.cache_scratch
export MPLCONFIGDIR=$XDG_CACHE_HOME/mpl
export TMPDIR=/n/lab_storage/eddy_lab/users/nhilgert/tmp
```